# Tarea 2 - vectorización y sentimiento

Voy a usar reseñas de medicamentos de Druglib que también tienen una calificación del 1 al 10

Primero haré la vectorización con TF-IDF y luego compararé AFINN con el rating

Fuentes

- [Druglib de UCI](https://archive.ics.uci.edu/dataset/461/drug+review+dataset+druglib)
- [AFINN](https://github.com/fnielsen/afinn)
- [TfidfVectorizer](https://scikit-learn.org/stable/modules/generated/sklearn.feature_extraction.text.TfidfVectorizer.html)
- [Similitud coseno](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.pairwise.cosine_similarity.html)

## Datos

Los datos vienen en un archivo zip de UCI

In [ ]:
import re
import urllib.request
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import nltk

from io import BytesIO
from zipfile import ZipFile
from nltk.corpus import stopwords
from nltk.stem import SnowballStemmer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

nltk.download('stopwords', quiet=True)

In [ ]:
url_datos = 'https://archive.ics.uci.edu/static/public/461/drug%2Breview%2Bdataset%2Bdruglib%2Bcom.zip'

# aquí descargo el zip y leo los dos archivos
respuesta = urllib.request.urlopen(url_datos, timeout=60)
comprimido = ZipFile(BytesIO(respuesta.read()))
archivos_datos = [nombre for nombre in comprimido.namelist() if nombre.lower().endswith('.tsv')]

partes = []
for nombre in archivos_datos:
    partes.append(pd.read_csv(comprimido.open(nombre), sep='\t'))

datos = pd.concat(partes, ignore_index=True)
comprimido.close()

datos.head()

## Limpieza inicial

Junto los tres comentarios y quito las filas que no tienen texto o rating

In [ ]:
datos['benefitsReview'] = datos['benefitsReview'].fillna('')
datos['sideEffectsReview'] = datos['sideEffectsReview'].fillna('')
datos['commentsReview'] = datos['commentsReview'].fillna('')

datos['resena'] = datos['benefitsReview'] + ' ' + datos['sideEffectsReview'] + ' ' + datos['commentsReview']
datos['resena'] = datos['resena'].str.strip()
datos['rating'] = pd.to_numeric(datos['rating'], errors='coerce')
datos = datos.dropna(subset=['rating'])
datos = datos[datos['resena'].str.len() > 0].reset_index(drop=True)

datos['cantidad_palabras'] = datos['resena'].str.split().str.len()
datos['cantidad_caracteres'] = datos['resena'].str.len()

datos[['rating', 'cantidad_palabras', 'cantidad_caracteres']].describe().round(2)

In [ ]:
datos['rating'].value_counts().sort_index().plot.bar()
plt.title('Cantidad de reseñas por rating')
plt.xlabel('rating')
plt.ylabel('cantidad')
plt.show()

datos['cantidad_palabras'].clip(upper=600).hist(bins=30)
plt.title('Palabras por reseña')
plt.xlabel('palabras')
plt.show()

## Limpiar el texto

Paso todo a minúsculas, quito palabras vacías y saco la raíz de cada palabra

In [ ]:
palabras_vacias = set(stopwords.words('english'))
raices = SnowballStemmer('english')

textos_limpios = []
for texto in datos['resena']:
    palabras = re.findall(r"[a-z]+(?:'[a-z]+)?", str(texto).lower())
    palabras = [palabra for palabra in palabras if palabra not in palabras_vacias and len(palabra) > 2]
    palabras = [raices.stem(palabra) for palabra in palabras]
    textos_limpios.append(' '.join(palabras))

datos['texto_limpio'] = textos_limpios
datos = datos[datos['texto_limpio'].str.len() > 0].reset_index(drop=True)

datos[['resena', 'texto_limpio', 'rating']].sample(3)

## TF-IDF

Uso palabras y grupos de dos con un máximo de 3000 términos

Después reviso el tamaño, los valores diferentes de cero y la densidad

In [ ]:
vectorizador = TfidfVectorizer(max_features=3000, min_df=3, ngram_range=(1, 2))
matriz_vectores = vectorizador.fit_transform(datos['texto_limpio'])
terminos = vectorizador.get_feature_names_out()

filas, columnas = matriz_vectores.shape
valores_no_cero = matriz_vectores.nnz
densidad = valores_no_cero / (filas * columnas)

matriz_vectores.shape, valores_no_cero, round(densidad, 4)

In [ ]:
promedios_tfidf = np.asarray(matriz_vectores.mean(axis=0)).ravel()
posiciones = np.argsort(promedios_tfidf)[-15:][::-1]
terminos_importantes = terminos[posiciones]
valores_importantes = promedios_tfidf[posiciones]

plt.barh(terminos_importantes[::-1], valores_importantes[::-1])
plt.xlabel('TF-IDF promedio')
plt.title('Palabras con mayor peso')
plt.show()

In [ ]:
# uso pocas reseñas para que se alcance a ver
indices_muestra = datos.sample(12).index.to_numpy()
matriz_similitud = cosine_similarity(matriz_vectores[indices_muestra])

plt.imshow(matriz_similitud)
plt.colorbar()
plt.title('Similitud entre algunas reseñas')
plt.show()

## Sentimiento con AFINN

AFINN le da un valor positivo o negativo a varias palabras en inglés

También saco el valor por cada 100 palabras para no favorecer a las reseñas largas

In [ ]:
url_afinn = 'https://raw.githubusercontent.com/fnielsen/afinn/master/afinn/data/AFINN-en-165.txt'
lexico = pd.read_csv(url_afinn, sep='\t', names=['termino', 'valor'])
valores_afinn = dict(zip(lexico['termino'], lexico['valor']))

sentimientos = []
sentimientos_normalizados = []

for texto in datos['resena']:
    palabras = re.findall(r"[a-z]+(?:'[a-z]+)?", str(texto).lower())
    total = 0
    for palabra in palabras:
        total = total + valores_afinn.get(palabra, 0)
    normalizado = total / max(len(palabras), 1) * 100
    sentimientos.append(total)
    sentimientos_normalizados.append(normalizado)

datos['sentimiento_total'] = sentimientos
datos['sentimiento_normalizado'] = sentimientos_normalizados

categorias_texto = []
for valor in datos['sentimiento_normalizado']:
    if valor < 0:
        categorias_texto.append('negativo')
    elif valor > 0:
        categorias_texto.append('positivo')
    else:
        categorias_texto.append('neutral')

datos['sentimiento_texto'] = categorias_texto
datos[['rating', 'sentimiento_total', 'sentimiento_normalizado', 'sentimiento_texto']].head()

## Comparación con el rating

Tomé del 1 al 4 como negativo, 5 y 6 como neutral y del 7 al 10 como positivo

Al final aparece primero la correlación y después la coincidencia

In [ ]:
categorias_rating = []
for rating in datos['rating']:
    if rating <= 4:
        categorias_rating.append('negativo')
    elif rating <= 6:
        categorias_rating.append('neutral')
    else:
        categorias_rating.append('positivo')

datos['sentimiento_rating'] = categorias_rating

correlacion = datos[['rating', 'sentimiento_normalizado']].corr(method='spearman').iloc[0, 1]
coincidencia = (
    datos['sentimiento_texto'].astype(str) == datos['sentimiento_rating'].astype(str)
).mean()

round(correlacion, 4), round(coincidencia, 4)

In [ ]:
promedio_por_rating = datos.groupby('rating')['sentimiento_normalizado'].mean()
promedio_por_rating.plot.bar()
plt.axhline(0, color='black')
plt.title('Sentimiento promedio por rating')
plt.ylabel('sentimiento')
plt.show()

## Conclusiones

La matriz de TF-IDF tiene muchos espacios en cero, algo normal porque no todas las palabras aparecen en cada reseña

AFINN sí se puede comparar con el rating pero no siempre coinciden, ya que una reseña puede hablar bien del medicamento y también mencionar efectos negativos

### Cosas a tomar en cuenta

- AFINN no entiende bien negaciones o expresiones médicas
- Yo junté beneficios, efectos secundarios y el comentario general
- Los cortes del rating los elegí para hacer esta comparación
- Esto solo analiza opiniones y no sirve para sacar conclusiones médicas